# LSTM 实验：在同一个记忆问题上替换 RNN cell

你在上一个实验里用手写 RNN 做了“读完整条序列、记住第一个信号”的任务。这个实验保持**数据、划分和训练循环不变**，只把 cell 从 RNN 换成 LSTM：新增 cell state $c_t$ 和三个门，手写 `lstm_step`，再和 `nn.LSTMCell` 核对。然后在 $T=10/30/100$ 上和 RNN 对比，并观察门值与 cell state 随时间的变化。

核心函数保留了完整签名、docstring 和 TODO 提示，函数体留给你实现。公式、API 链接、自检单元格和全部绘图代码已经提供。

## 使用方法

1. 在本目录运行 `uv sync`，然后在 VS Code / Jupyter 中选择 `.venv/bin/python` 作为 kernel。
2. 从上往下运行。遇到 `TODO` 函数时先实现它，再运行紧随其后的自检单元格。
3. 修改函数后要重新运行定义它的单元格，编辑器里的代码不会自动同步到 kernel。
4. 没实现的函数会抛出 `NotImplementedError`，这是练习刚开始时的正常状态。
5. 保持函数名、参数名、返回值约定和属性名不变（`self.gates`、`self.out`），自检和绘图依赖它们。
6. 全部完成后执行 Restart Kernel → Run All，确认结果不依赖旧变量。

**规则**：LSTM 的递推要自己写，不要在模型里调用 `nn.LSTM`、`nn.LSTMCell`。`nn.LSTMCell` 只出现在自检里，用来核对你的 `lstm_step`。

## 路线

| Lab | 内容 | 你要实现 | 已提供的观察图 |
| --- | --- | --- | --- |
| 1 | LSTM 的一步和整个序列 | `LSTMClassifier.__init__`, `lstm_step`, `forward` | 计算流程图 |
| 2 | 与 `nn.LSTMCell` 核对 | `copy_weights_to_lstmcell` | 无 |
| 3 | 实验：T=10 / 30 / 100，对比 RNN | 无（运行并记录） | 准确率与 loss 曲线 |
| 4 | 梯度：cell state 的加法路径 | 无（运行并记录） | 各时间步的梯度范数 |
| 5 | 观察门值和 cell state | `gate_trace_to_numpy` | 门值、cell state、hidden state 随时间变化 |

一共 5 个函数 / 方法。数据生成、训练循环和 RNN 对照模型直接复用上一个实验，已经提供。

## 1. Problem Definition

和上一个实验完全相同：

- 每个样本是长度 $T$ 的序列，第一个数字随机取 $-1$ 或 $+1$，其余 $T-1$ 个数字独立采样自 $\text{Uniform}(-1,1)$。
- 网络读完整条序列，用最后的状态判断第一个数字的正负。

| 项目 | 本实验的约定 |
| --- | --- |
| 输入 | `[B, T, 1]` |
| 标签 | `[B, 1]`，浮点数 0 或 1 |
| 隐藏状态 $h_t$ 和 cell state $c_t$ | 都是 `[B, 32]` |
| 输出 | `[B, 1]`，一个 logit |
| 数据量 | 10,000 训练 / 2,000 验证 / 2,000 测试，独立生成 |
| 损失 | `BCEWithLogitsLoss` |
| 训练配置 | Adam，`lr=0.001`，batch 64，20 轮，梯度裁剪 1.0 |

**要回答的问题**：

1. 你的 LSTM 实现对吗？（和 `nn.LSTMCell` 核对）
2. 把 RNN cell 换成 LSTM cell 之后，$T=30$ 和 $T=100$ 会更好吗？
3. LSTM 的哪个设计让梯度更容易传到很早的时间步？门值和 cell state 在记住第一个数字时做了什么？

> **关于结果的预期**：LSTM 的长序列表现是一个**实验观察**，不是保证。训练结果会受初始化、学习率和训练预算影响。你得到的结果可能和“LSTM 总是比 RNN 好”的说法不一样，那正是这个实验想让你看到的。以你自己运行的结果为准，如实记录。

## 2. 环境与 API 速查

下表只列出实现所需的**部分参数**，不是 API 的全部参数。训练流程的 API 和上一个实验相同，不再重复。

### 2.1 LSTM 的实现（Lab 1、2）

| API（官方文档） | 关键参数 / 返回值 | 用在哪里 |
| --- | --- | --- |
| [`nn.Linear(in_features, out_features)`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html) | `weight` 形状 `[out, in]`，`bias` 形状 `[out]`；计算 $xW^\top+b$ | 把 4 个门的线性变换合成一次矩阵乘 |
| [`torch.cat(tensors, dim=1)`](https://docs.pytorch.org/docs/stable/generated/torch.cat.html) | 沿已有维度拼接：`[B, I]` 和 `[B, H]` → `[B, I+H]` | 拼出 $[x_t,h_{t-1}]$ |
| [`Tensor.chunk(chunks, dim=1)`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.chunk.html) | 沿 `dim` 切成等份；`[B, 4H]` → 4 个 `[B, H]` | 切出 $i,f,g,o$ 的预激活 |
| [`torch.sigmoid(x)`](https://docs.pytorch.org/docs/stable/generated/torch.sigmoid.html)、[`torch.tanh(x)`](https://docs.pytorch.org/docs/stable/generated/torch.tanh.html) | 逐元素；sigmoid 输出在 $(0,1)$，tanh 输出在 $(-1,1)$ | 门与候选信息 |
| [`nn.init.uniform_(tensor, a, b)`](https://docs.pytorch.org/docs/stable/nn.init.html#torch.nn.init.uniform_) | 原地用 $U(a,b)$ 填充 | 初始化 |
| [`nn.LSTMCell(input_size, hidden_size)`](https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTMCell.html) | `weight_ih` `[4H, I]`、`weight_hh` `[4H, H]`、`bias_ih`、`bias_hh` `[4H]`；**门的顺序是 i, f, g, o**；`cell(x, (h, c))` 返回 `(h', c')` | 只在自检里核对 `lstm_step` |

### 2.2 观察与训练（Lab 3–5）

| API（官方文档） | 关键参数 / 返回值 | 用在哪里 |
| --- | --- | --- |
| [`Tensor.retain_grad()`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.retain_grad.html) | 让非叶子张量保留 `.grad` | 观察 $\partial L/\partial c_t$、$\partial L/\partial h_t$ |
| [`Tensor.detach()`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.detach.html)、[`Tensor.numpy()`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.numpy.html) | 脱离计算图；转成 NumPy（必须在 CPU 上、不带梯度） | 画图 |
| [`torch.stack(tensors, dim=0)`](https://docs.pytorch.org/docs/stable/generated/torch.stack.html) | 沿**新**维度堆叠：$T$ 个 `[1, H]` → `[T, 1, H]` | 整理轨迹 |
| `nn.utils.clip_grad_norm_`、`BCEWithLogitsLoss`、`Adam` | 同 RNN 实验 | 训练 |

官方参考：[`nn.LSTMCell`](https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTMCell.html)（公式和状态定义）、[`nn.LSTM`](https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html)。

> **TODO（研究）**：打开 `nn.LSTMCell` 文档的公式部分，把它和第 4 节的公式逐项对照。文档里的 $W_{ii}, W_{hi}$ 对应本实验的 `gates.weight` 的哪一块？

### 2.3 NumPy 手写、PyTorch、TensorFlow / Keras 对比

本实验用 PyTorch 手写 LSTM。NumPy 版和 TensorFlow / Keras 只做对照，不运行。

| 维度 | NumPy 手写（含 BPTT） | PyTorch 手写 `lstm_step`（本实验） | PyTorch `nn.LSTM` / `nn.LSTMCell` | TensorFlow / Keras `LSTM` |
| --- | --- | --- | --- | --- |
| 前向 | 6 个公式，用 `np.tanh` 和手写 sigmoid | 同左，换成 `torch` 张量运算 | `nn.LSTM` 是融合 kernel（cuDNN 等），无 Python 循环；`nn.LSTMCell` 只是单步，时间循环仍要自己写 | 同左 |
| 反向传播 | 要手推 4.4 的全部公式并存下 $i,f,g,o,c,h$ | autograd 自动完成 | 自动完成 | 自动完成 |
| 4 个门的权重 | 自定 | 一个 `nn.Linear(I+H, 4H)`，列 $[:I]$ 作用在 $x_t$，列 $[I:]$ 作用在 $h_{t-1}$ | `weight_ih` + `weight_hh` 分开存，顺序 **i, f, g, o** | `kernel` `[I, 4H]` + `recurrent_kernel` `[H, 4H]`，顺序 **i, f, c(g), o** |
| bias | 自定 | 一个 `bias`，形状 `[4H]` | **两个**：`bias_ih`、`bias_hh`（数学上等价于一个） | 一个，形状 `[4H]` |
| forget gate 偏置初值 | 自定 | `forget_bias` 参数，默认 0 | **没有**特殊处理，和其他参数一样随机 | `unit_forget_bias=True`（**默认**）：初始化为 1 |
| 取出中间量 | 自己存 | 随手可取（本实验的 `return_trace`） | `nn.LSTM` 只返回输出和最后的 $(h,c)$；要每步的门值得自己拆成 `LSTMCell` 循环 | 同样不直接给门值 |
| 速度（本实验规模） | 最慢 | 慢 | `nn.LSTM` 快，`nn.LSTMCell` 循环和手写版接近 | 快 |

**取舍**：手写版适合看清每个门，也是唯一能直接取出门值的做法。训练真实任务时用融合实现。Keras 默认就把 forget gate 的偏置设成 1，PyTorch 不会，这个差别会在第 3 节的实验里影响结果。

> **TODO（研究）**：
> 1. 把 `nn.LSTMCell` 的 `bias_ih`、`bias_hh` 合并成一个 bias 后，forget gate 那一段对应 `bias[H:2H]`。验证 Lab 2 的权重复制里为什么可以把 `bias_hh` 置零。
> 2. 用 Keras 写同一个任务，打开 `unit_forget_bias=False` 对比，看结果是否和第 3 节相符。

In [ ]:
import copy
import time
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)
print("Matplotlib:", matplotlib.__version__)

SERIES_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#4a3aa7"]
HILITE = "#eb6834"

plt.rcParams["font.sans-serif"] = [
    "PingFang SC", "Heiti SC", "Arial Unicode MS", "Microsoft YaHei", "SimHei",
    "Noto Sans CJK SC", "WenQuanYi Micro Hei", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 11,
})

DEVICE = torch.device("cpu")     # 序列模型的张量很小，CPU 比 GPU / MPS 更快
torch.set_num_threads(4)

## 3. 核心参数配置

| Hyperparameter | 代码常量 | 值 | 说明 |
| --- | --- | --- | --- |
| 对比的序列长度 | `T_LIST` | `[10, 30, 100]` | |
| 数据量 | `N_TRAIN`, `N_VAL`, `N_TEST` | 10,000 / 2,000 / 2,000 | |
| 输入 / 隐藏 / 输出维度 | `INPUT_SIZE`, `HIDDEN_SIZE`, `OUTPUT_SIZE` | 1 / 32 / 1 | |
| Learning rate | `LEARNING_RATE` | 0.001 | Adam |
| Batch size / Epochs | `BATCH_SIZE`, `EPOCHS` | 64 / 20 | |
| 梯度裁剪阈值 | `CLIP_NORM` | 1.0 | |
| Forget gate 偏置对比值 | `FORGET_BIAS_ON` | 1.0 | 基线用 0.0 |
| Random seed | `SEED` | 42 | |

LSTM 的参数量：$4H(I+H)+4H+H\cdot O+O=4\cdot32\cdot33+128+32+1=4385$，大约是 RNN（1121）的 3.9 倍，因为有 4 组线性变换。

In [ ]:
T_LIST = [10, 30, 100]
N_TRAIN, N_VAL, N_TEST = 10_000, 2_000, 2_000
INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE = 1, 32, 1
LEARNING_RATE = 1e-3
BATCH_SIZE = 64
EPOCHS = 20
CLIP_NORM = 1.0
FORGET_BIAS_ON = 1.0
SEED = 42

n_params_lstm = 4 * HIDDEN_SIZE * (INPUT_SIZE + HIDDEN_SIZE) + 4 * HIDDEN_SIZE + HIDDEN_SIZE * OUTPUT_SIZE + OUTPUT_SIZE
print("LSTM 可训练参数数量：", n_params_lstm)
assert n_params_lstm == 4385

## 4. 公式总表

### 4.1 符号与形状

采用**行向量**写法（每一行是一个样本），所以线性变换是 $zW^\top$。课程里常见的列向量写法 $W[x_t,h_{t-1}]$ 对应的实际 batch 代码就是下面的形式，只是矩阵多了一次转置。

| 数学符号 | 代码变量 | Shape | 含义 |
| --- | --- | --- | --- |
| $x_t$ | `x[:, t]` | $(B, I)$ | 第 $t$ 步输入 |
| $h_{t-1},\,c_{t-1}$ | `h_prev`, `c_prev` | $(B, H)$ | 上一步的隐藏状态和 cell state，$h_0=c_0=\mathbf 0$ |
| $z_t=[x_t,h_{t-1}]$ | `z` | $(B, I+H)$ | 沿特征维度拼接 |
| $W=[W_i;W_f;W_g;W_o]$ | `self.gates.weight` | $(4H, I+H)$ | 4 个门的权重沿行方向堆叠，顺序 **i, f, g, o** |
| $b=[b_i;b_f;b_g;b_o]$ | `self.gates.bias` | $(4H,)$ | 对应的偏置 |
| $i_t,f_t,g_t,o_t$ | `i, f, g, o` | $(B, H)$ | 输入门、遗忘门、候选信息、输出门 |
| $c_t,\,h_t$ | `c`, `h` | $(B, H)$ | 新的 cell state 和隐藏状态 |
| $W_y,\,b_y$ | `self.out.weight`, `self.out.bias` | $(O,H)$, $(O,)$ | 输出层 |

### 4.2 LSTM 的一步

先做一次矩阵乘，再把结果沿特征维度切成 4 份：

$$
[a_i,\;a_f,\;a_g,\;a_o]=z_tW^\top+b,\qquad z_t=[x_t,\;h_{t-1}]
$$

$$
\begin{aligned}
i_t &= \sigma(a_i) & f_t &= \sigma(a_f)\\
g_t &= \tanh(a_g) & o_t &= \sigma(a_o)\\
c_t &= f_t\odot c_{t-1}+i_t\odot g_t\\
h_t &= o_t\odot\tanh(c_t)
\end{aligned}
$$

它们与课程公式 $i_t=\sigma(W_i[x_t,h_{t-1}]+b_i)$ 等逐项对应，只是 4 个门放进了同一个大矩阵。

| 变量 | 作用 |
| --- | --- |
| $f_t$ | 控制旧 cell state **保留**多少（接近 1 = 全部保留，接近 0 = 清空） |
| $i_t$ | 控制候选信息**写入**多少 |
| $g_t$ | 当前产生的候选信息，取值在 $(-1,1)$ |
| $o_t$ | 控制 cell state 向 hidden state **输出**多少 |
| $c_t$ | 沿时间传递的记忆状态 |
| $h_t$ | 供后续计算和预测使用的隐藏状态 |

整个序列：$h_0=c_0=\mathbf 0$，循环 $t=1,\dots,T$ 之后 $z=h_TW_y^\top+b_y$，损失和预测与 RNN 实验相同（`BCEWithLogitsLoss`，$\hat y=\mathbf 1[z>0]$）。**每个 batch 开始时 $h_0$ 和 $c_0$ 都要重新置零**。

### 4.3 参数量

$$
\#\text{params}=\underbrace{4H(I+H)+4H}_{\text{gates}}+\underbrace{HO+O}_{\text{output}}
$$

### 4.4 BPTT（供推导和 NumPy 手写参考）

记 $\delta h_t=\partial\mathcal L/\partial h_t$，$\delta c_t=\partial\mathcal L/\partial c_t$（都是**总梯度**，包含来自后面时间步的部分）。已知 $\delta h_t$ 和从未来传来的 $\delta c_t^{\text{future}}=\delta c_{t+1}\odot f_{t+1}$ 之后：

$$
\begin{aligned}
\delta o_t&=\delta h_t\odot\tanh(c_t)\\
\delta c_t&=\delta c_t^{\text{future}}+\delta h_t\odot o_t\odot\bigl(1-\tanh^2(c_t)\bigr)\\
\delta i_t&=\delta c_t\odot g_t,\quad \delta f_t=\delta c_t\odot c_{t-1},\quad \delta g_t=\delta c_t\odot i_t\\
\delta c_{t-1}^{\text{future}}&=\delta c_t\odot f_t
\end{aligned}
$$

通过激活函数回到预激活 $a$（$\sigma'(a)=\sigma(a)(1-\sigma(a))$，$\tanh'(a)=1-\tanh^2(a)$）：

$$
\delta a_i=\delta i_t\odot i_t(1-i_t),\quad
\delta a_f=\delta f_t\odot f_t(1-f_t),\quad
\delta a_g=\delta g_t\odot(1-g_t^2),\quad
\delta a_o=\delta o_t\odot o_t(1-o_t)
$$

拼成 $\delta a_t=[\delta a_i,\delta a_f,\delta a_g,\delta a_o]\in\mathbb R^{B\times4H}$，则

$$
\delta z_t=\delta a_t\,W\ \ (\text{前 }I\text{ 列是 }\delta x_t,\ \text{后 }H\text{ 列是 }\delta h_{t-1}\text{ 的一部分}),\qquad
\frac{\partial\mathcal L}{\partial W}=\sum_{t=1}^T\delta a_t^{\top}z_t,\qquad
\frac{\partial\mathcal L}{\partial b}=\sum_{t=1}^T\sum_{n=1}^B\delta a_t
$$

和 RNN 一样，共享参数 $W,b$ 的梯度是所有时间步贡献之和。

### 4.5 cell state 的加法路径

$c_t=f_t\odot c_{t-1}+i_t\odot g_t$ 是 $c_{t-1}\to c_t$ 的**加法更新**。如果先只看这条直接路径（忽略门通过 $h_{t-1}$ 对 $c_{t-1}$ 的间接依赖）：

$$
\frac{\partial c_t}{\partial c_{t-1}}\Bigg|_{\text{direct}}=\mathrm{diag}(f_t),\qquad
\frac{\partial c_T}{\partial c_k}\Bigg|_{\text{direct}}=\mathrm{diag}\Bigl(\prod_{t=k+1}^{T}f_t\Bigr)
$$

对比 RNN：$\partial h_t/\partial h_{t-1}=\mathrm{diag}(1-h_t^2)\,W_h$，每一步都要乘一次矩阵 $W_h$ 和一个 $\le1$ 的因子，连乘后容易指数消失或爆炸。

LSTM 的这条路径上**没有**权重矩阵，也**没有** tanh 的导数，只乘逐元素的门值 $f_t\in(0,1)$：

- $f_t\approx1$ 时，梯度几乎无损地传回早期时间步（这就是 “constant error carousel”）。
- $f_t$ 如果只有 0.5，连乘 29 步后只剩 $0.5^{29}\approx2\times10^{-9}$。`forget_bias=1` 时初始 $f\approx\sigma(1)\approx0.73$，$0.73^{29}\approx1\times10^{-4}$；`forget_bias=3` 时 $f\approx0.95$，$0.95^{29}\approx0.24$。所以**遗忘门偏置的初值**会明显改变初始的梯度大小。
- 但这只是一个**解释候选，不是证明**：门值依赖输入和 $h_{t-1}$，Adam 会重新缩放梯度，训练中 $f_t$ 也在变。$0.73^{29}$ 仍然很小，而 `forget_bias=1` 在实验里 $T=30$ 依然学会了，说明光看这个连乘不够。

这条路径是“直接路径”：完整的梯度还包含通过 $h_{t-1}$ 和各个门的间接项，那些项仍然会经过矩阵乘法。加法路径让长距离梯度**有机会**保持，但不保证训练一定成功。

In [ ]:
# 参考图：LSTM 一步的数据流
def draw_lstm_cell():
    fig, ax = plt.subplots(figsize=(11, 5.6))
    ax.set_xlim(0, 12); ax.set_ylim(0, 6.4); ax.axis("off"); ax.grid(False)

    def box(x, y, text, color, w=1.3, h=0.7, fs=10, fc="white"):
        ax.add_patch(FancyBboxPatch((x - w / 2, y - h / 2), w, h, boxstyle="round,pad=0.04", fc=fc, ec=color, lw=2))
        ax.text(x, y, text, ha="center", va="center", fontsize=fs)

    def arrow(p, q, color="#555", lw=1.6, ls="-", rad=0.0):
        ax.add_patch(FancyArrowPatch(p, q, arrowstyle="-|>", mutation_scale=13, color=color, lw=lw, ls=ls,
                                     connectionstyle=f"arc3,rad={rad}", shrinkA=1, shrinkB=1))

    # cell state 主干（顶部，粗线）
    box(0.6, 5.5, r"$c_{t-1}$", SERIES_COLORS[0], w=0.9)
    box(11.3, 5.5, r"$c_t$", SERIES_COLORS[0], w=0.9)
    arrow((1.1, 5.5), (3.4, 5.5), SERIES_COLORS[0], lw=3)
    box(3.8, 5.5, r"$\odot$", SERIES_COLORS[1], w=0.6, h=0.6, fs=13)           # 乘 f
    arrow((4.15, 5.5), (6.9, 5.5), SERIES_COLORS[0], lw=3)
    box(7.3, 5.5, r"$+$", SERIES_COLORS[2], w=0.6, h=0.6, fs=13)               # 加 i*g
    arrow((7.65, 5.5), (10.8, 5.5), SERIES_COLORS[0], lw=3)
    ax.text(5.6, 6.05, "cell state 的加法路径：梯度可以沿这条线几乎无损地传回去", ha="center", fontsize=10, color=SERIES_COLORS[0])
    # 门（4 个门由同一个 z_t 得到）
    ys = 3.2
    # i 和 g 先相乘再加到主干
    box(5.4, ys, r"$i_t=\sigma(a_i)$" + "\n输入门", SERIES_COLORS[2], w=1.5, h=0.9, fs=9)
    box(7.3, ys + 1.3, r"$\odot$", SERIES_COLORS[2], w=0.5, h=0.5, fs=12)
    box(7.4, ys, r"$g_t=\tanh(a_g)$" + "\n候选信息", SERIES_COLORS[3], w=1.5, h=0.9, fs=9)
    box(3.8, ys, r"$f_t=\sigma(a_f)$" + "\n遗忘门", SERIES_COLORS[1], w=1.6, h=0.9, fs=9)
    box(9.6, ys, r"$o_t=\sigma(a_o)$" + "\n输出门", SERIES_COLORS[4], w=1.6, h=0.9, fs=9)
    arrow((3.8, ys + 0.45), (3.8, 5.2), SERIES_COLORS[1])                     # f -> ⊙(旧 c)
    arrow((5.4, ys + 0.45), (7.05, ys + 1.3), SERIES_COLORS[2], rad=-0.15)    # i -> ⊙
    arrow((7.4, ys + 0.45), (7.35, ys + 1.05), SERIES_COLORS[3])               # g -> ⊙
    arrow((7.3, ys + 1.55), (7.3, 5.2), SERIES_COLORS[2])                     # ⊙ -> +
    # 从 c_t 取 tanh 再乘 o 得到 h_t
    box(9.6, 4.6, r"$\tanh$", "#777", w=0.8, h=0.5)
    arrow((9.6, 5.45), (9.6, 4.85), "#777", rad=0.0)
    box(10.9, 3.2, r"$\odot$", SERIES_COLORS[4], w=0.5, h=0.5, fs=12)
    arrow((9.6, 4.35), (10.75, 3.4), "#777", rad=0.1)
    arrow((10.4, ys), (10.65, ys), SERIES_COLORS[4])
    box(10.9, 1.4, r"$h_t$", SERIES_COLORS[0], w=0.9)
    arrow((10.9, 2.95), (10.9, 1.75), SERIES_COLORS[0])
    # 输入
    box(0.6, 1.4, r"$h_{t-1}$", SERIES_COLORS[0], w=0.9)
    box(0.6, 0.45, r"$x_t$", "#888", w=0.9)
    box(2.9, 0.9, r"$z_t=[x_t,h_{t-1}]$" + "\n" + r"$a=z_tW^\top+b$", "#888", w=2.4, h=0.9, fs=9)
    arrow((1.05, 1.4), (1.7, 1.05), "#888"); arrow((1.05, 0.45), (1.7, 0.75), "#888")
    for gx in (3.8, 5.4, 7.4, 9.6):
        arrow((4.1, 1.1), (gx, ys - 0.45), "#aaa", lw=1.0, ls="--", rad=0.0)
    ax.set_title("LSTM 一步：4 个门由同一个 $z_t$ 经同一次矩阵乘得到，主干上只有逐元素的乘和加", fontsize=11.5)
    plt.show()


draw_lstm_cell()

> **你来做（手画）**：合上上面的图，在纸上画出 $T=3$ 的 LSTM 展开图，有两条状态线：$h_t$ 和 $c_t$。然后用红笔标出 $\mathcal L$ 到 $c_1$ 的梯度路径，数一数它经过了几个 $\odot f_t$（$T=3$ 时应该是 2 个）。回答：这条路径上有没有 $W$？
>
> 你的回答：

## 5. 复用上一个实验的代码（已提供）

数据生成、DataLoader、训练和评估和上一个实验**完全相同**。你可以直接使用下面的版本，也可以换成你自己在 RNN 实验里写的实现（接口一致）。同时提供一个 `VanillaRNN` 对照模型，它的递推与上一个实验的 `SimpleRNN` 相同，`forward` 多了一个 `return_trace` 参数，方便和 LSTM 用同一个接口观察梯度。

> `return_trace=True` 时模型返回 `(logits, trace)`，`trace` 是字典，每个值是长度为 $T$ 的 **list**（元素 `(B, H)`，是计算图上的原始张量）。RNN 只有键 `"h"`；LSTM 有 `"h","c","i","f","g","o"`。

In [ ]:
def generate_sequences(n, T, seed):
    """第一个数字为 ±1，其余为 Uniform(-1, 1) 干扰；标签是第一个数字是否为正。"""
    g = torch.Generator().manual_seed(seed)
    X = torch.rand(n, T, 1, generator=g) * 2 - 1
    first = torch.randint(0, 2, (n,), generator=g) * 2 - 1
    X[:, 0, 0] = first.float()
    y = (first > 0).float().unsqueeze(1)
    return X, y


def make_loaders(T, seed=SEED, batch_size=BATCH_SIZE):
    data = {}
    for offset, (name, n) in enumerate([("train", N_TRAIN), ("val", N_VAL), ("test", N_TEST)]):
        data[name] = generate_sequences(n, T, seed=seed * 1000 + offset)
    train_loader = DataLoader(TensorDataset(*data["train"]), batch_size=batch_size, shuffle=True,
                              generator=torch.Generator().manual_seed(seed))
    val_loader = DataLoader(TensorDataset(*data["val"]), batch_size=batch_size)
    test_loader = DataLoader(TensorDataset(*data["test"]), batch_size=batch_size)
    return train_loader, val_loader, test_loader, data


def train_epoch(model, loader, loss_fn, optimizer, clip_norm):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    grad_norms = []
    for x, y in loader:
        logits = model(x)
        loss = loss_fn(logits, y)
        optimizer.zero_grad()
        loss.backward()
        grad_norms.append(nn.utils.clip_grad_norm_(model.parameters(), clip_norm).item())   # 返回裁剪前的范数
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        correct += ((logits > 0).float() == y).sum().item()
        total += x.size(0)
    return total_loss / total, correct / total, grad_norms


def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            logits = model(x)
            total_loss += loss_fn(logits, y).item() * x.size(0)
            correct += ((logits > 0).float() == y).sum().item()
            total += x.size(0)
    return total_loss / total, correct / total


class VanillaRNN(nn.Module):
    """上一个实验的 RNN（对照用）。h_t = tanh(x_t W_x^T + h_{t-1} W_h^T + b)。"""

    def __init__(self, input_size=INPUT_SIZE, hidden_size=HIDDEN_SIZE, output_size=OUTPUT_SIZE):
        super().__init__()
        self.hidden_size = hidden_size
        k = 1.0 / np.sqrt(hidden_size)
        self.W_x = nn.Parameter(torch.empty(hidden_size, input_size).uniform_(-k, k))
        self.W_h = nn.Parameter(torch.empty(hidden_size, hidden_size).uniform_(-k, k))
        self.b = nn.Parameter(torch.zeros(hidden_size))
        self.W_y = nn.Parameter(torch.empty(output_size, hidden_size).uniform_(-k, k))
        self.b_y = nn.Parameter(torch.zeros(output_size))

    def forward(self, x, return_trace=False):
        h = x.new_zeros(x.size(0), self.hidden_size)
        states = []
        for t in range(x.size(1)):
            h = torch.tanh(x[:, t] @ self.W_x.T + h @ self.W_h.T + self.b)
            states.append(h)
        logits = h @ self.W_y.T + self.b_y
        return (logits, {"h": states}) if return_trace else logits


def state_grad_norms(model, x, y, loss_fn):
    """每个时间步的 ||dL/dh_t||（LSTM 还有 ||dL/dc_t||），单样本 loss，batch 内平均。返回 {键: (T,) 张量}。"""
    logits, trace = model(x, return_trace=True)
    keys = [k for k in ("h", "c") if k in trace]
    for k in keys:
        for s in trace[k]:
            s.retain_grad()
    model.zero_grad()
    loss_fn(logits, y).backward()
    B = x.size(0)      # loss 取了 batch 平均，梯度带有 1/B，乘回来得到单样本的梯度
    return {k: torch.stack([(s.grad * B).norm(dim=1).mean() for s in trace[k]]).detach() for k in keys}


print("已加载：generate_sequences, make_loaders, train_epoch, evaluate, VanillaRNN, state_grad_norms")

## Lab 1：实现 LSTM 的一步和整个序列

**你要实现** `LSTMClassifier` 的三个部分：

| 部分 | 要求 |
| --- | --- |
| `__init__(input_size, hidden_size, output_size, forget_bias=0.0)` | `self.gates = nn.Linear(input_size + hidden_size, 4 * hidden_size)`，`self.out = nn.Linear(hidden_size, output_size)`。所有权重用 $U(-1/\sqrt H,\,1/\sqrt H)$ 均匀初始化，所有 bias 先置 0，**然后把 `gates.bias[H:2H]`（遗忘门那一段）设成 `forget_bias`**。 |
| `lstm_step(x_t, h_prev, c_prev, return_gates=False)` | 4.2 的一步。返回 `(h_t, c_t)`；`return_gates=True` 时返回 `(h_t, c_t, (i, f, g, o))`。 |
| `forward(x, return_trace=False)` | `x` 是 `(B, T, I)`。**在函数内部**把 $h_0$、$c_0$ 置零，循环 $T$ 步，用最后的 $h_T$ 过 `self.out` 得到 logits `(B, O)`。`return_trace=True` 时返回 `(logits, trace)`，`trace` 是字典，键 `"h","c","i","f","g","o"`，每个值是长度 $T$ 的 list（元素 `(B, H)`，不要 detach）。 |

**提示**：

- 先 `z = torch.cat([x_t, h_prev], dim=1)`，再 `self.gates(z).chunk(4, dim=1)` 得到 `a_i, a_f, a_g, a_o`，**顺序是 i, f, g, o**。
- `i, f, o` 用 `torch.sigmoid`，`g` 用 `torch.tanh`。
- `c = f * c_prev + i * g`，`h = o * torch.tanh(c)`，注意用逐元素乘 `*`，不是矩阵乘 `@`。
- 零状态：`x.new_zeros(B, H)`。
- 不要在循环里 `.detach()`；不要把状态保存在 `self` 上。

In [ ]:
class LSTMClassifier(nn.Module):
    """手写的 many-to-one LSTM 分类器。

    Input : x [B, T, input_size]
    Output: logits [B, output_size]
    """

    def __init__(
        self,
        input_size: int = INPUT_SIZE,
        hidden_size: int = HIDDEN_SIZE,
        output_size: int = OUTPUT_SIZE,
        forget_bias: float = 0.0,
    ) -> None:
        # TODO: super().__init__()；保存 self.input_size、self.hidden_size；创建 self.gates 和 self.out。
        # TODO: 权重 U(-k, k)，k = 1 / sqrt(hidden_size)；bias 全 0；
        #       然后 gates.bias[hidden_size : 2 * hidden_size] = forget_bias（在 torch.no_grad() 里改）。
        raise NotImplementedError("TODO Lab 1: implement LSTMClassifier.__init__")

    def lstm_step(self, x_t, h_prev, c_prev, return_gates: bool = False):
        """4.2 的一步。x_t: (B, I)，h_prev, c_prev: (B, H)。

        Returns
        -------
        (h_t, c_t) 或 (h_t, c_t, (i, f, g, o))，所有张量形状 (B, H)。
        """
        # TODO: z = cat；chunk 成 4 份；i/f/o 用 sigmoid，g 用 tanh；c = f*c_prev + i*g；h = o*tanh(c)。
        raise NotImplementedError("TODO Lab 1: implement LSTMClassifier.lstm_step")

    def forward(self, x: torch.Tensor, return_trace: bool = False):
        """读完整个序列，用最后的 h 输出 logits。"""
        # TODO: h、c 置零；trace = {k: [] for k in "h c i f g o".split()}；
        #       for t in range(T): 调用 lstm_step(..., return_gates=True)，把每步的 h,c,i,f,g,o 追加进 trace。
        # TODO: logits = self.out(h)。
        raise NotImplementedError("TODO Lab 1: implement LSTMClassifier.forward")

In [ ]:
# Lab 1 自检 A：参数形状、初始化、forget_bias
torch.manual_seed(SEED)
lstm = LSTMClassifier()
shapes = {name: tuple(p.shape) for name, p in lstm.named_parameters()}
assert shapes == {"gates.weight": (128, 33), "gates.bias": (128,), "out.weight": (1, 32), "out.bias": (1,)}, f"参数形状不对：{shapes}"
assert sum(p.numel() for p in lstm.parameters()) == 4385
k = 1 / np.sqrt(32)
assert lstm.gates.weight.abs().max() <= k + 1e-6 and lstm.gates.weight.abs().max() > 0.5 * k, "权重应来自 U(-1/sqrt(H), 1/sqrt(H))"
assert (lstm.gates.bias == 0).all() and (lstm.out.bias == 0).all(), "forget_bias=0 时所有 bias 都应为 0"
lstm_fb = LSTMClassifier(forget_bias=1.0)
b = lstm_fb.gates.bias.detach()
assert (b[32:64] == 1.0).all(), "遗忘门（第二段 [H:2H]）的 bias 应等于 forget_bias"
assert (b[:32] == 0).all() and (b[64:] == 0).all(), "其他门的 bias 应为 0"
assert lstm_fb.gates.bias.requires_grad, "bias 仍然要可训练（改值时要在 torch.no_grad() 里）"

# Lab 1 自检 B：lstm_step —— 两个可以心算的特例
H_ = 2
tiny = LSTMClassifier(input_size=1, hidden_size=H_, output_size=1)
with torch.no_grad():
    tiny.gates.weight.zero_()
    # 特例 1：f≈1, i≈0, o≈1 —— 旧记忆原样保留，新信息不写入
    tiny.gates.bias.copy_(torch.tensor([-20, -20, 20, 20, 0, 0, 20, 20], dtype=torch.float32))   # [i | f | g | o]
c_prev = torch.tensor([[0.5, -1.0]])
h1, c1 = tiny.lstm_step(torch.tensor([[0.3]]), torch.tensor([[0.1, 0.2]]), c_prev)
np.testing.assert_allclose(c1.detach().numpy(), c_prev.numpy(), atol=1e-6)
np.testing.assert_allclose(h1.detach().numpy(), np.tanh(c_prev.numpy()), atol=1e-6)
with torch.no_grad():
    # 特例 2：f≈0, i≈1, g=tanh(0.5), o≈1 —— 清空旧记忆，写入新信息
    tiny.gates.bias.copy_(torch.tensor([20, 20, -20, -20, 0.5, 0.5, 20, 20], dtype=torch.float32))
h2, c2 = tiny.lstm_step(torch.tensor([[0.3]]), torch.tensor([[0.1, 0.2]]), c_prev)
np.testing.assert_allclose(c2.detach().numpy(), np.tanh([[0.5, 0.5]]), atol=1e-6)
np.testing.assert_allclose(h2.detach().numpy(), np.tanh(np.tanh([[0.5, 0.5]])), atol=1e-6)
# 返回约定
h, c = lstm.lstm_step(torch.randn(3, 1), torch.zeros(3, 32), torch.zeros(3, 32))
assert h.shape == c.shape == (3, 32)
out = lstm.lstm_step(torch.randn(3, 1), torch.zeros(3, 32), torch.zeros(3, 32), return_gates=True)
assert len(out) == 3 and len(out[2]) == 4
# 随机权重：手动按 [x_t, h_{t-1}] 的拼接顺序和 i, f, g, o 的分块顺序算一遍，能发现 cat 顺序或门顺序写反
m_r = LSTMClassifier(input_size=2, hidden_size=3, output_size=1)
with torch.no_grad():
    m_r.gates.weight.copy_(torch.randn(12, 5)); m_r.gates.bias.copy_(torch.randn(12))
xr, hr, cr = torch.randn(4, 2), torch.randn(4, 3), torch.randn(4, 3)
W_, b_ = m_r.gates.weight.detach(), m_r.gates.bias.detach()
a_ = xr @ W_[:, :2].T + hr @ W_[:, 2:].T + b_            # 前 2 列作用在 x_t，后 3 列作用在 h_{t-1}
ai, af, ag, ao = a_[:, 0:3], a_[:, 3:6], a_[:, 6:9], a_[:, 9:12]
c_exp = torch.sigmoid(af) * cr + torch.sigmoid(ai) * torch.tanh(ag)
h_exp = torch.sigmoid(ao) * torch.tanh(c_exp)
h_got, c_got = m_r.lstm_step(xr, hr, cr)
torch.testing.assert_close(c_got, c_exp, atol=1e-6, rtol=1e-5)
torch.testing.assert_close(h_got, h_exp, atol=1e-6, rtol=1e-5)
print("Lab 1 自检 A、B 通过：参数、forget_bias、两个特例、随机权重的手算对照都正确。")

In [ ]:
# Lab 1 自检 C：门值范围；forward 的形状、trace、状态独立性、梯度连通
xb = torch.randn(5, 12, 1)
logits, trace = lstm_fb(xb, return_trace=True)
assert logits.shape == (5, 1)
assert set(trace) == {"h", "c", "i", "f", "g", "o"}
assert all(len(v) == 12 and all(s.shape == (5, 32) for s in v) for v in trace.values())
for name in ("i", "f", "o"):
    v = torch.stack(trace[name])
    assert (v > 0).all() and (v < 1).all(), f"{name} 门应该用 sigmoid，取值在 (0, 1)"
assert (torch.stack(trace["g"]).abs() < 1).all(), "候选信息 g 应该用 tanh，取值在 (-1, 1)"
assert (torch.stack(trace["h"]).abs() < 1).all()
torch.testing.assert_close(logits, lstm_fb.out(trace["h"][-1]))
torch.testing.assert_close(lstm_fb(xb), logits)                                  # 重复调用结果相同（状态不跨调用）
torch.testing.assert_close(lstm_fb(xb[2:3]), lstm_fb(xb)[2:3], atol=1e-6, rtol=1e-5)   # 每个样本从零状态开始
assert not hasattr(lstm_fb, "h") and not hasattr(lstm_fb, "c"), "不要把状态存在模型对象里"
# 状态更新关系在整条轨迹上成立：c_t = f_t ⊙ c_{t-1} + i_t ⊙ g_t，h_t = o_t ⊙ tanh(c_t)
c_prev_ = torch.zeros(5, 32)
for t in range(12):
    torch.testing.assert_close(trace["c"][t], trace["f"][t] * c_prev_ + trace["i"][t] * trace["g"][t], atol=1e-6, rtol=1e-5)
    torch.testing.assert_close(trace["h"][t], trace["o"][t] * torch.tanh(trace["c"][t]), atol=1e-6, rtol=1e-5)
    c_prev_ = trace["c"][t]
# 梯度连通：loss 对第一步输入有梯度，没有 detach
xg = xb.clone().requires_grad_(True)
lstm_fb(xg).sum().backward()
assert xg.grad[:, 0].abs().sum() > 0, "第一个时间步收不到梯度：循环里是不是 detach 了？"
assert all(torch.isfinite(p.grad).all() for p in lstm_fb.parameters())
print("Lab 1 自检 C 通过：门值范围、状态关系、独立性和梯度连通都正确。")

## Lab 2：把同一组权重复制到 `nn.LSTMCell`，核对单步输出

**你要实现**：`copy_weights_to_lstmcell(model)`：创建 `nn.LSTMCell(input_size, hidden_size)`，把 `model.gates` 的权重复制进去，返回这个 cell。

对应关系（门的顺序两边都是 **i, f, g, o**）：

| `nn.LSTMCell` | 来自 `model.gates` |
| --- | --- |
| `weight_ih`，形状 `(4H, I)` | `weight[:, :I]`（作用在 $x_t$ 上的那些列） |
| `weight_hh`，形状 `(4H, H)` | `weight[:, I:]`（作用在 $h_{t-1}$ 上的那些列） |
| `bias_ih`，形状 `(4H,)` | `bias` |
| `bias_hh`，形状 `(4H,)` | 全 0（`LSTMCell` 有两个 bias，它们相加才是有效 bias，我们只用一个） |

改参数要放在 `torch.no_grad()` 里，用 `copy_`。

In [ ]:
def copy_weights_to_lstmcell(model: LSTMClassifier) -> nn.LSTMCell:
    """返回一个权重与 model.gates 等价的 nn.LSTMCell。"""
    # TODO: cell = nn.LSTMCell(model.input_size, model.hidden_size)；
    #       no_grad 里 copy weight_ih / weight_hh / bias_ih；bias_hh 置 0。
    raise NotImplementedError("TODO Lab 2: implement copy_weights_to_lstmcell")

In [ ]:
# Lab 2 自检：单步和整条序列都和 nn.LSTMCell 一致（bias 用非零随机值，更容易暴露顺序错误）
torch.manual_seed(3)
m = LSTMClassifier(forget_bias=0.5)
with torch.no_grad():
    m.gates.bias.add_(torch.randn_like(m.gates.bias) * 0.3)
cell = copy_weights_to_lstmcell(m)
assert isinstance(cell, nn.LSTMCell)
assert cell.weight_ih.shape == (128, 1) and cell.weight_hh.shape == (128, 32)
assert torch.equal(cell.bias_hh, torch.zeros(128))

x_t, h_p, c_p = torch.randn(7, 1), torch.randn(7, 32), torch.randn(7, 32)
h_mine, c_mine = m.lstm_step(x_t, h_p, c_p)
h_ref, c_ref = cell(x_t, (h_p, c_p))
torch.testing.assert_close(h_mine, h_ref, atol=1e-6, rtol=1e-5)
torch.testing.assert_close(c_mine, c_ref, atol=1e-6, rtol=1e-5)

# 整条序列：每一步的 h、c 都核对
x_seq = torch.randn(4, 15, 1)
_, tr = m(x_seq, return_trace=True)
h_r = c_r = torch.zeros(4, 32)
for t in range(15):
    h_r, c_r = cell(x_seq[:, t], (h_r, c_r))
    torch.testing.assert_close(tr["h"][t], h_r, atol=1e-5, rtol=1e-4, msg=lambda s: f"第 {t + 1} 步 h 不一致\n{s}")
    torch.testing.assert_close(tr["c"][t], c_r, atol=1e-5, rtol=1e-4, msg=lambda s: f"第 {t + 1} 步 c 不一致\n{s}")
print("Lab 2 自检通过：手写 LSTM 与 nn.LSTMCell 在单步和 15 步序列上完全一致。")

## Lab 3：在 T = 10 / 30 / 100 上训练，并和 RNN 对比

下面的实验已提供，复用上面的 `train_epoch` 和 `evaluate`。对每个序列长度**从头训练三个模型**，用同一份数据和同一个种子：

| 模型 | 说明 |
| --- | --- |
| `RNN` | `VanillaRNN`，和上一个实验相同 |
| `LSTM (forget_bias=0)` | 你的 `LSTMClassifier`，遗忘门偏置 0（初始 $f\approx0.5$） |
| `LSTM (forget_bias=1)` | 同上，遗忘门偏置 1（初始 $f\approx0.73$） |

**验收**：$T=10$ 上两个 LSTM 的验证准确率都超过 95%。$T=30$ 和 $T=100$ **不设通过线**，是观察记录。

> 先别急着看结果，写下你的预测：你认为 T=30、T=100 上谁会学会？LSTM 一定比 RNN 好吗？
>
> 你的预测：

In [ ]:
def run_experiment(make_model, T, label, epochs=EPOCHS, seed=SEED, clip_norm=CLIP_NORM, verbose_every=10):
    train_loader, val_loader, test_loader, _ = make_loaders(T, seed=seed)
    torch.manual_seed(seed)
    model = make_model()
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    hist = {"train_loss": [], "val_acc": [], "grad_norm": []}
    t0 = time.time()
    for epoch in range(1, epochs + 1):
        tr_loss, _, norms = train_epoch(model, train_loader, loss_fn, optimizer, clip_norm)
        _, va_acc = evaluate(model, val_loader, loss_fn)
        hist["train_loss"].append(tr_loss); hist["val_acc"].append(va_acc); hist["grad_norm"].append(float(np.mean(norms)))
        if epoch % verbose_every == 0 or epoch == 1:
            print(f"{label:<22} T={T:>3} epoch {epoch:>2}/{epochs}  train loss {tr_loss:.4f}  val acc {va_acc:.4f}  ({time.time() - t0:.0f}s)")
    hist["test_acc"] = evaluate(model, test_loader, loss_fn)[1]
    return hist, model


MODELS = {
    "RNN": lambda: VanillaRNN(),
    "LSTM (forget_bias=0)": lambda: LSTMClassifier(forget_bias=0.0),
    "LSTM (forget_bias=1)": lambda: LSTMClassifier(forget_bias=FORGET_BIAS_ON),
}
results, trained = {}, {}
for T in T_LIST:
    for label, factory in MODELS.items():
        results[(label, T)], trained[(label, T)] = run_experiment(factory, T, label)
    print()

for label in ("LSTM (forget_bias=0)", "LSTM (forget_bias=1)"):
    assert results[(label, 10)]["val_acc"][-1] > 0.95, f"{label} 在 T=10 的验证准确率应超过 95%，回头检查 lstm_step / forward"
print("验收：T=10 上两个 LSTM 都通过。")

In [ ]:
# 观察图：每个序列长度一张子图，三个模型的验证准确率
epochs_axis = np.arange(1, EPOCHS + 1)
style = {"RNN": ("-", "o", SERIES_COLORS[0]), "LSTM (forget_bias=0)": ("--", "s", SERIES_COLORS[1]), "LSTM (forget_bias=1)": ("-.", "^", SERIES_COLORS[2])}
fig, axes = plt.subplots(2, 3, figsize=(15, 7.2), sharex=True)
for col, T in enumerate(T_LIST):
    for label in MODELS:
        ls, mk, color = style[label]
        axes[0, col].plot(epochs_axis, results[(label, T)]["val_acc"], ls, marker=mk, ms=4, color=color, label=label)
        axes[1, col].plot(epochs_axis, results[(label, T)]["train_loss"], ls, marker=mk, ms=4, color=color, label=label)
    axes[0, col].axhline(0.5, color="#999", lw=1, ls=":"); axes[0, col].set_ylim(0.45, 1.02)
    axes[0, col].set_title(f"T={T}：验证准确率")
    axes[1, col].axhline(np.log(2), color="#999", lw=1, ls=":"); axes[1, col].set_title(f"T={T}：训练 loss（虚线 = ln 2 随机猜）")
    axes[1, col].set_xlabel("epoch"); axes[1, col].set_xticks(range(2, EPOCHS + 1, 2))
axes[0, 0].legend(loc="lower right"); axes[0, 0].set_ylabel("accuracy"); axes[1, 0].set_ylabel("BCE loss")
fig.tight_layout(); plt.show()

print(f"{'模型':<24}{'T':>5}{'最终验证准确率':>14}{'测试准确率':>12}{'首次验证准确率 > 0.9 的 epoch':>30}")
for T in T_LIST:
    for label in MODELS:
        r = results[(label, T)]
        first = next((i + 1 for i, a in enumerate(r["val_acc"]) if a > 0.9), None)
        print(f"{label:<24}{T:>5}{r['val_acc'][-1]:>14.4f}{r['test_acc']:>12.4f}{str(first):>30}")

### 记录你的观察

**以你自己运行的结果为准**，如实记录，即使它和“LSTM 总比 RNN 好”的预期不符。

| T | 模型 | 最终验证准确率 | 首次 > 0.9 的 epoch | 训练难度（容易 / 有 plateau / 学不会） |
| --- | --- | --- | --- | --- |
| 10 | RNN | | | |
| 10 | LSTM (fb=0) | | | |
| 10 | LSTM (fb=1) | | | |
| 30 | RNN | | | |
| 30 | LSTM (fb=0) | | | |
| 30 | LSTM (fb=1) | | | |
| 100 | RNN | | | |
| 100 | LSTM (fb=0) | | | |
| 100 | LSTM (fb=1) | | | |

**问题（写在这里）**

1. 上一个实验里你记录的 RNN 数字和这里的 RNN 一致吗？（前提是你的 `generate_sequences` 与 `SimpleRNN.__init__` 的随机数调用顺序和提示一致；顺序不同，数据或初始化就不同，数字会有差别，这不是 bug）
2. T=30 上，forget_bias=0 和 forget_bias=1 的 LSTM 有区别吗？用 4.5 的 $\prod f_t$ 和 Lab 4 的梯度比值提出一个解释，再想想这个解释哪里不够（见 4.5 最后一条）。
3. T=100 上谁学会了？如果都没学会，你认为是初始化、学习率还是训练预算的问题？下面的研究清单里有可以尝试的改动。
4. 这些结果能不能说明“LSTM 比 RNN 更适合长序列”？还需要补充什么实验才能下结论？（提示：换随机种子、多跑几次、改学习率。）

你的回答：

## Lab 4：cell state 的加法路径与梯度

下面比较**未训练**的模型在 $T=30$ 序列上，各个时间步的梯度范数（代码已提供，使用 `state_grad_norms`）：

- RNN 的 $\|\partial\mathcal L/\partial h_t\|$。
- LSTM 的 $\|\partial\mathcal L/\partial c_t\|$，分别用 `forget_bias=0, 1, 3` 初始化。

按 4.5，cell state 这条路径的直接衰减率是 $f_t$ 的连乘。`forget_bias` 越大，初始 $f$ 越接近 1，梯度应该越容易传回早期时间步。**纵轴是对数坐标**。

In [ ]:
Xg, yg = generate_sequences(256, 30, seed=7)
bce = nn.BCEWithLogitsLoss()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.4))
steps = np.arange(1, 31)
torch.manual_seed(0)
g_rnn = state_grad_norms(VanillaRNN(), Xg, yg, bce)["h"]
ax1.plot(steps, g_rnn.numpy(), "-", color=SERIES_COLORS[0], lw=2, label=r"RNN：$\|\partial L/\partial h_t\|$")
summary = {"RNN h": g_rnn}
for fb, ls, color in [(0.0, "--", SERIES_COLORS[1]), (1.0, "-.", SERIES_COLORS[2]), (3.0, ":", SERIES_COLORS[3])]:
    torch.manual_seed(0)
    g = state_grad_norms(LSTMClassifier(forget_bias=fb), Xg, yg, bce)
    ax1.plot(steps, g["c"].numpy(), ls, color=color, lw=2, label=rf"LSTM fb={fb:g}：$\|\partial L/\partial c_t\|$")
    ax2.plot(steps, g["h"].numpy(), ls, color=color, lw=2, label=rf"LSTM fb={fb:g}：$\|\partial L/\partial h_t\|$")
    summary[f"LSTM fb={fb:g} c"] = g["c"]
ax2.plot(steps, g_rnn.numpy(), "-", color=SERIES_COLORS[0], lw=2, label=r"RNN：$\|\partial L/\partial h_t\|$")
for ax, title in [(ax1, "cell state 路径（LSTM）与 RNN 的隐藏状态梯度"), (ax2, "hidden state 路径")]:
    ax.set_yscale("log"); ax.set_xlabel("时间步 t（30 = 最后一步，1 = 第一个数字所在的步）")
    ax.set_ylabel("梯度范数（对数坐标）"); ax.set_title(title); ax.legend(fontsize=8)
fig.tight_layout(); plt.show()

print("第 1 步与第 30 步的梯度范数之比（越接近 1，梯度越没衰减）：")
for name, g in summary.items():
    print(f"  {name:<18}{(g[0] / g[-1]).item():.3e}")

> **你来做（解释）**：
>
> 1. 上图中 `forget_bias=0` 的 LSTM 的 cell state 梯度，和 RNN 的相比，哪个衰减得更快？这和你在 Lab 3 里看到的训练结果一致吗？
> 2. `forget_bias` 变大，曲线怎样变化？用 4.5 的 $\prod f_t$ 解释。
> 3. 为什么 cell state 的加法路径“对梯度传递有利”？用 $\partial c_t/\partial c_{t-1}|_{\text{direct}}=\mathrm{diag}(f_t)$ 对比 RNN 的 $\mathrm{diag}(1-h_t^2)W_h$。
> 4. 加法路径是**直接路径**。完整的 $\partial c_t/\partial c_{t-1}$ 里还有哪些项？它们为什么没有消除问题，只是减轻？
>
> 你的回答：

## Lab 5：观察门值和 cell state 随时间的变化

用 Lab 3 中训练好的 `LSTM (forget_bias=1)`（$T=30$）。选两个样本：一个第一个数字是 $+1$，一个是 $-1$，看部分隐藏单元的 $i_t,f_t,o_t,c_t,h_t$ 怎么变化。

**你要实现**：`gate_trace_to_numpy(trace)`：把 `forward(x, return_trace=True)` 返回的 `trace`（字典，每个值是长度 $T$ 的 list，元素 `(1, H)`）整理成字典，每个值是 NumPy 数组，形状 `(T, H)`。要求：`detach`，不带梯度。

**提示**：`torch.stack(list, dim=0)` 得到 `(T, 1, H)`，`.squeeze(1)` 去掉 batch 维，`.detach().numpy()`。

In [ ]:
def gate_trace_to_numpy(trace: dict[str, list[torch.Tensor]]) -> dict[str, np.ndarray]:
    """把单个样本的 trace 转成 {键: ndarray (T, H)}。"""
    # TODO: 对每个键，torch.stack → squeeze(1) → detach → numpy。
    raise NotImplementedError("TODO Lab 5: implement gate_trace_to_numpy")

In [ ]:
# Lab 5 自检
m = LSTMClassifier(forget_bias=1.0)
x1 = torch.randn(1, 9, 1)
with torch.no_grad():
    _, tr1 = m(x1, return_trace=True)
arr = gate_trace_to_numpy(tr1)
assert set(arr) == {"h", "c", "i", "f", "g", "o"}
assert all(isinstance(v, np.ndarray) and v.shape == (9, 32) for v in arr.values())
_, tr_g = m(x1, return_trace=True)                    # 带梯度的 trace 也必须能转换
arr_g = gate_trace_to_numpy(tr_g)
np.testing.assert_allclose(arr_g["c"][4], tr_g["c"][4].detach().numpy()[0])
print("Lab 5 gate_trace_to_numpy 自检通过。")

In [ ]:
# 观察图：同一个模型读正样本和负样本，门值和 cell state 如何变化（T=30，forget_bias=1 的 LSTM）
model_obs = trained[("LSTM (forget_bias=1)", 30)]
model_obs.eval()
Xo, yo = generate_sequences(512, 30, seed=2024)
with torch.no_grad():
    _, tr_all = model_obs(Xo, return_trace=True)
cT = tr_all["c"][-1].numpy()
pos = yo[:, 0] == 1
# 选出最后一步的 cell state 在正 / 负样本上差别最大的 4 个隐藏单元：它们最可能携带“第一个数字的符号”
gap = np.abs(cT[pos.numpy()].mean(0) - cT[~pos.numpy()].mean(0))
units = np.argsort(-gap)[:4]
print("选中的隐藏单元：", units.tolist(), "（最后一步 c 在正 / 负样本上的均值差最大）")

idx_pos, idx_neg = int(torch.where(pos)[0][0]), int(torch.where(~pos)[0][0])
rows = [("x", "输入 x_t"), ("i", "输入门 i_t"), ("f", "遗忘门 f_t"), ("o", "输出门 o_t"), ("c", "cell state c_t"), ("h", "hidden state h_t")]
fig, axes = plt.subplots(len(rows), 2, figsize=(13, 13), sharex=True)
for col, (idx, title) in enumerate([(idx_pos, "正样本（第一个数字 +1）"), (idx_neg, "负样本（第一个数字 -1）")]):
    with torch.no_grad():
        _, tr = model_obs(Xo[idx:idx + 1], return_trace=True)
    arr = gate_trace_to_numpy(tr)
    arr["x"] = Xo[idx, :, 0].numpy()[:, None]
    for row, (key, name) in enumerate(rows):
        ax = axes[row, col]
        if key == "x":
            ax.plot(range(1, 31), arr["x"][:, 0], "-o", color="#999", ms=3.5, lw=1)
            ax.plot([1], [arr["x"][0, 0]], "o", color=HILITE, ms=10, zorder=5)
            ax.axhline(0, color="#bbb", lw=0.8)
        else:
            for u, color, ls in zip(units, SERIES_COLORS, ["-", "--", "-.", ":"]):
                ax.plot(range(1, 31), arr[key][:, u], ls, color=color, lw=1.8, label=f"单元 {u}")
        if key in ("i", "f", "o"):
            ax.set_ylim(-0.02, 1.02)
        ax.set_ylabel(name, fontsize=9)
        if row == 0:
            ax.set_title(title)
    axes[-1, col].set_xlabel("时间步 t")
handles, labels = axes[1, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=4, bbox_to_anchor=(0.5, 0.965), fontsize=9, frameon=False)
fig.suptitle("门值和 cell state 随时间的变化（橙色圆点 = 第一个数字）", y=0.995)
fig.tight_layout(rect=[0, 0, 1, 0.95]); plt.show()

# 同一批单元：正 / 负样本的 c_t 平均轨迹（512 个样本）
fig, axes = plt.subplots(1, 4, figsize=(15, 3.4), sharey=False)
for ax, u in zip(axes, units):
    c_traj = torch.stack(tr_all["c"])[:, :, u].numpy()     # (T, N)
    ax.plot(range(1, 31), c_traj[:, pos.numpy()].mean(1), "-", color=SERIES_COLORS[1], lw=2, label="正样本均值")
    ax.plot(range(1, 31), c_traj[:, ~pos.numpy()].mean(1), "--", color=SERIES_COLORS[0], lw=2, label="负样本均值")
    ax.fill_between(range(1, 31), c_traj[:, pos.numpy()].mean(1) - c_traj[:, pos.numpy()].std(1),
                    c_traj[:, pos.numpy()].mean(1) + c_traj[:, pos.numpy()].std(1), color=SERIES_COLORS[1], alpha=0.12)
    ax.fill_between(range(1, 31), c_traj[:, ~pos.numpy()].mean(1) - c_traj[:, ~pos.numpy()].std(1),
                    c_traj[:, ~pos.numpy()].mean(1) + c_traj[:, ~pos.numpy()].std(1), color=SERIES_COLORS[0], alpha=0.12)
    ax.set_title(f"单元 {u} 的 c_t"); ax.set_xlabel("时间步 t")
axes[0].legend(fontsize=8)
fig.suptitle("512 个样本的 c_t 平均轨迹（阴影 = ±1 标准差）：正负样本的轨迹何时分开、能否一直保持分开？", y=1.03)
fig.tight_layout(); plt.show()
print("最后一步这 4 个单元的 c_T：正样本均值", cT[pos.numpy()][:, units].mean(0).round(2), " 负样本均值", cT[~pos.numpy()][:, units].mean(0).round(2))

> **你来做（观察与解释）**：
>
> 1. 第一个数字出现在 $t=1$。在 $t=1$ 附近，输入门 $i_t$ 和 $c_t$ 发生了什么？是“t=1 一次写入，之后原样保持”吗？还是别的方式？
> 2. 在 $t=2\dots30$ 的干扰期，同一个单元的 $f_t$ 在正样本和负样本上一样吗？不同单元的 $f_t$ 一样吗？门值是常数吗？门的输入里有什么会随时间变化？（提示：4.2 里 $a=[x_t,h_{t-1}]W^\top+b$，门是 $h_{t-1}$ 的函数。）
> 3. $c_t$ 在干扰期是平稳不变，还是持续变化？有的单元 $c_T$ 的符号和第一个数字相关，有的没有，你怎么看？训练后的网络是不是一定采用“写一次、保持不变”这种最容易理解的方式？
> 4. 如果你用 `forget_bias=0`、没有学会的 T=30 模型（或 T=100 的模型）画同样的图，轨迹会有什么不同？
>
> 你的回答：

## 6. 为什么门用 Sigmoid，候选信息用 Tanh？

> **你来做（解释）**：用你自己的话回答，写完后可以和下面的提示对照。
>
> 1. 门 $i_t,f_t,o_t$ 的作用是“按比例放行”，需要什么样的取值范围？Sigmoid 的输出范围、单调性和导数分别有什么性质？如果换成 ReLU 或 Tanh 做门，会出什么问题？
> 2. 候选信息 $g_t$ 要写进 cell state，为什么需要正负都能取，并且有界？如果换成 Sigmoid 会怎样？如果换成 ReLU 呢？
> 3. $c_t$ 本身没有被任何激活函数压缩，为什么 $h_t=o_t\odot\tanh(c_t)$ 里还要再 `tanh` 一次？
>
> 你的回答：
>
> **提示**：门的值在 $(0,1)$ 才可以解释为“比例”；$g_t\in(-1,1)$ 让 cell state 可以被加也可以被减，且每一步的写入量有上界；$c_t$ 是一个累加的量，没有上界。

## 进阶研究清单

> **TODO（研究）**
>
> - [ ] **遗忘门偏置扫描**：把 `forget_bias` 取 0、1、2、3、5 各训练一次 T=30 / T=100，画出最终准确率。Keras 的 `unit_forget_bias=True` 是这个实验的一个特例。
> - [ ] **T=100 的训练预算**：把 `EPOCHS` 增加到 100，或把学习率改成 `3e-3` / `1e-2`，T=100 能学会吗？记录第一次学会的 epoch。
> - [ ] **多个随机种子**：把 `SEED` 换成 0–4 各跑一次，统计每种模型学会的次数。这决定了你能否对“LSTM 比 RNN 好”下结论。
> - [ ] **正交初始化**：给 RNN 的 $W_h$ 使用 `nn.init.orthogonal_`，T=100 有改善吗？和 LSTM 的 `forget_bias` 对比。
> - [ ] **GRU**：只用 2 个门（reset、update）实现 `GRUClassifier`，参数量比 LSTM 少多少？在同样的任务上表现如何？
> - [ ] **Peephole LSTM**：让门也看到 $c_{t-1}$。对这个任务有帮助吗？
> - [ ] **NumPy 版 BPTT**：按 4.4 的公式，用 NumPy 写 `lstm_backward`，用 `torch.autograd` 校验每个参数的梯度。这是理解 LSTM 反向传播最直接的方式。
> - [ ] **融合实现**：把 `LSTMClassifier` 换成 `nn.LSTM(1, 32, batch_first=True)` + `nn.Linear`，比较速度。用 `nn.LSTM` 能取出门值吗？
> - [ ] **TensorFlow / Keras 对照**：见 2.3 的 TODO。
> - [ ] **阅读**：[Hochreiter & Schmidhuber, *Long Short-Term Memory*, 1997](https://www.bioinf.jku.at/publications/older/2604.pdf)（原始论文，constant error carousel）、Gers, Schmidhuber & Cummins, *Learning to Forget: Continual Prediction with LSTM*, Neural Computation, 2000（遗忘门的引入；请自己搜索原文）、[Jozefowicz et al., *An Empirical Exploration of Recurrent Network Architectures*, 2015](https://proceedings.mlr.press/v37/jozefowicz15.html)（遗忘门偏置初值 1 的经验结论）、[Understanding LSTM Networks (Olah, 2015)](https://colah.github.io/posts/2015-08-Understanding-LSTMs/)。
> - [ ] **从这里往后**：注意力机制 / Transformer 怎样绕开“必须沿时间一步步传梯度”的限制？见本仓库 `03_llm/02_transformer_from_scratch/`。